# M3 — EDA: bid book, volumes and no-buy patterns (mFRR)

Owner: Iustin (M3, learning bidders). Sprint 1 cards:
- *EDA: bid book, volumes and no-buy patterns*
- *Prototype descriptive acceptance/rejection reaction analysis*

Jean Coupon (18 Nov): start with the **weekly** auctions; check whether the bid statistics change **before a spike**.

Sections
1. Bid-book statistics per auction
2. Weekly spikes: definition (spike-and-decay shape; price/regime EDA is Carolina's card)
3. Event study: bid book before, at and after the start of a spike
4. Accepted vs rejected bids
5. Can we follow a bidder? (proxy feasibility for Week 2)
6. No-buy auctions
7. Summary and limitations

Everything "before a spike" uses only auctions whose result is published before the spike auction's gate closure (weekly: previous week's result is known 1 week before).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

from src.data.data import build_auctions

OFFERS = ROOT / "data/raw/offers"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

# palette: categorical slots 1-2 + neutral (dataviz default palette)
C1, C2, GREY, INK = "#2a78d6", "#eb6834", "#9a9893", "#52514e"
DIR_COLOR = {"downwards": C1, "upwards": C2}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6, "lines.linewidth": 2,
    "legend.frameon": False,
})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

## 1. Bid-book statistics per auction

In [ ]:
offers = pd.read_parquet(OFFERS, filters=[("auction_type", "==", "mFRR")])
offers = offers[~offers["is_anticipated"]].copy()
auctions = build_auctions(str(OFFERS), product="mFRR")

mp = offers[offers["accepted"]].groupby("auction_id")["power_price"].max().rename("mp")
offers = offers.join(mp, on="auction_id")
offers["rel"] = np.log10(offers["power_price"] / offers["mp"])   # log10(bid / marginal)


def book_stats(o: pd.DataFrame) -> pd.DataFrame:
    acc, rej = o[o["accepted"]], o[~o["accepted"]]
    near = o[(o["rel"].abs() <= np.log10(1.1))]                       # within ±10 % of marginal
    s = pd.DataFrame({
        "n_acc": acc.groupby("auction_id").size(),
        "n_rej": rej.groupby("auction_id").size(),
        "acc_p10": acc.groupby("auction_id")["power_price"].quantile(0.1),
        "min_rej_above": rej[rej["power_price"] > rej["mp"]].groupby("auction_id")["power_price"].min(),
        "vol_near": near.groupby("auction_id")["offered_volume"].sum(),
    })
    return s


A = auctions.merge(book_stats(offers), left_on="auction_id", right_index=True, how="left")
A[["n_acc", "n_rej", "vol_near"]] = A[["n_acc", "n_rej", "vol_near"]].fillna(0)
A["cover"] = A["offered"] / A["awarded"].replace(0, np.nan)       # offered / awarded
A["rej_share_vol"] = 1 - A["awarded"] / A["offered"]
A["spread_acc"] = np.log10(A["marginal"] / A["acc_p10"])           # spread of accepted prices
A["gap_above"] = np.log10(A["min_rej_above"] / A["marginal"])      # headroom to next rejected bid
A["near_share"] = A["vol_near"] / A["offered"]

# weekly auctions, without the one 2019 correction auction (KORR)
W = A[(A["freq"] == "weekly") & ~A["auction_id"].str.contains("KORR")].sort_values("date")
STATS = ["n_bids", "offered", "awarded", "cover", "rej_share_vol", "n_rej",
         "near_share", "spread_acc", "gap_above"]
W.groupby("direction")[STATS].median().T.round(2)

Yearly medians of the weekly bid book. The **cover ratio** (offered / awarded) is the competition measure.

In [ ]:
yearly = W.groupby([W["date"].dt.year, "direction"])[["n_bids", "offered", "awarded", "cover"]].median()
yearly.unstack().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for ax, col, title in zip(axes, ["n_bids", "awarded", "cover"],
                          ["Bids per auction", "Awarded volume (MW)", "Cover ratio (offered / awarded)"]):
    for d, g in W.groupby("direction"):
        r = g.set_index("date")[col].rolling(8, min_periods=4).median()
        ax.plot(r.index, r.values, color=DIR_COLOR[d], label=d)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
axes[0].legend()
fig.suptitle("Weekly mFRR bid book, 8-week rolling median", x=0.01, ha="left", fontsize=11)
fig.tight_layout(); save(fig, "m3_weekly_book_over_time")


## 2. Weekly spikes: definition

Jean (18 Nov) describes the weekly price as **spikes and decays**. A spike is therefore defined by its **shape**, per direction:

1. **Fast rise:** the price reaches a local peak at least **0.2 log10 (×1.58)** above its lowest point of the previous **4 weeks**;
2. **Decay:** within the next **8 weeks** it falls back by at least **30 % of that rise** (it does not have to return to the start: prices often settle higher);
3. **One spike per episode:** peaks less than 4 weeks apart are merged (the higher one is kept).

Each spike has a **peak** (●, the spike as the eye sees it) and an **onset** (○, the low point where the rise starts).
The analysis "before a spike" is aligned on the **first week of the rise** (onset + 1).

*Not a definition from the brief or the Swissgrid documents: our choice, to be confirmed with Jean. An earlier rule ("+41 % vs last week") marked slow climbs several times and missed rises built over a few weeks.*


In [ ]:

from scipy.signal import find_peaks

RISE, UP_WEEKS, DECAY_WEEKS, DECAY_FRAC, MERGE_WEEKS = 0.2, 4, 8, 0.3, 4


def spike_shape(y: pd.Series) -> tuple[list[int], list[int]]:
    # positions of spike peaks and of their onsets (start of the rise) in the weekly series y
    v = y.interpolate().to_numpy()
    candidates, _ = find_peaks(v, distance=3)          # local maxima
    peaks, onsets = [], []
    for p in candidates:
        lo = max(0, p - UP_WEEKS)
        start = lo + int(np.argmin(v[lo:p]))             # lowest point in the weeks before the peak
        rise = v[p] - v[start]
        if rise < RISE:
            continue                                      # not a fast enough rise
        after = v[p + 1:p + 1 + DECAY_WEEKS]
        if len(after) == 0 or v[p] - after.min() < DECAY_FRAC * rise:
            continue                                      # no decay afterwards
        if peaks and p - peaks[-1] <= MERGE_WEEKS:        # same episode: keep the higher peak
            if v[p] > v[peaks[-1]]:
                peaks[-1], onsets[-1] = p, start
            continue
        peaks.append(p)
        onsets.append(start)
    return peaks, onsets


def weekly_panel(direction: str) -> pd.DataFrame:
    g = W[W["direction"] == direction].set_index("date").asfreq("7D")
    g["dy"] = g["y"].diff()
    peaks, onsets = spike_shape(g["y"])
    pos = lambda idx: pd.Series(np.isin(np.arange(len(g)), idx), index=g.index)
    g["peak"] = pos(peaks)
    g["onset"] = pos(onsets)
    g["rise_start"] = pos([min(s + 1, p) for s, p in zip(onsets, peaks)])   # first week of the rise
    episode = np.zeros(len(g), dtype=bool)
    for s, p in zip(onsets, peaks):
        episode[s:p + 2] = True                                             # onset .. peak + 1 week
    g["in_episode"] = episode
    g["rise"] = np.nan
    g.loc[g["peak"], "rise"] = [g["y"].iloc[p] - g["y"].iloc[s] for s, p in zip(onsets, peaks)]
    return g


panels = {d: weekly_panel(d) for d in ["downwards", "upwards"]}
pd.DataFrame({d: {"weeks": len(g), "spikes": int(g["peak"].sum()),
                  "median rise (log10)": round(g["rise"].median(), 2),
                  "median weeks onset → peak": float(np.median(np.diff(np.c_[np.flatnonzero(g["onset"]), np.flatnonzero(g["peak"])], axis=1))),
                  "median weekly change": round(g["dy"].median(), 3)}
              for d, g in panels.items()})


Between spikes the median weekly change is slightly **negative** (slow decay); spikes are the sudden rises.

In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(13, 5.5), sharex=True)
for ax, (d, g) in zip(axes, panels.items()):
    ax.plot(g.index, g["y"], color=DIR_COLOR[d], linewidth=1.5)
    ax.scatter(g.index[g["peak"]], g.loc[g["peak"], "y"], s=30, color=INK, zorder=3, label="spike (peak)")
    ax.scatter(g.index[g["onset"]], g.loc[g["onset"], "y"], s=28, facecolors="none", edgecolors=INK,
               zorder=3, label="rise starts")
    ax.set_title(f"Weekly {d}: log10 marginal price (CHF/MW per week) — {int(g['peak'].sum())} spikes",
                 loc="left", fontsize=10, color=INK)
axes[0].legend(loc="upper left")
fig.tight_layout(); save(fig, "m3_weekly_spikes")


In [ ]:

def half_life(g: pd.DataFrame, horizon: int = 12) -> pd.Series:
    # weeks after the peak until the price has given back half of the rise
    out = []
    for t in g.index[g["peak"]]:
        peak, rise = g.at[t, "y"], g.at[t, "rise"]
        fut = g["y"].loc[t:].iloc[1:horizon + 1]
        back = fut[fut <= peak - rise / 2]
        out.append((back.index[0] - t).days // 7 if len(back) else np.nan)
    return pd.Series(out, dtype=float)

pd.DataFrame({d: {"median weeks to give back half of the rise": half_life(g).median(),
                  "share that gives back half within 12 weeks": round(half_life(g).notna().mean(), 2)}
              for d, g in panels.items()})



## 3. Event study: the bid book around a spike

**Week 0 = the first week of the rise** (onset + 1). Weeks −1, −2, … are published before week 0's auction closes → usable for forecasting.

Each statistic is a deviation from its own recent level: the median of the 8 weeks ending at week −3 (log10 for counts and volumes),
so long-run trends (e.g. more bidders in 2024 than in 2019) are not mistaken for a spike signal.
We compare spikes with **normal weeks** = weeks outside any spike episode (onset → peak + 1).


In [ ]:

LOG = ["n_bids", "offered", "awarded", "cover"]
LAGS = range(-4, 5)   # k: statistic of week t+k, t = first week of the rise


def event_table(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    for c in LOG:
        g[c] = np.log10(g[c])
    normal = ~g["in_episode"] & g["dy"].notna()
    rows = []
    for c in STATS + ["y"]:
        base = g[c].shift(3).rolling(8, min_periods=5).median()
        for k in LAGS:
            dev = g[c].shift(-k) - base
            rows.append({"stat": c, "k": k,
                         "spike": dev[g["rise_start"]].median(), "normal": dev[normal].median()})
    return pd.DataFrame(rows)

ev = pd.concat({d: event_table(g) for d, g in panels.items()}, names=["direction"]).reset_index(level=0)
ev[ev["k"].isin([-2, -1, 0])].pivot_table(index="stat", columns=["direction", "k"], values=["spike", "normal"]).round(3)


In [ ]:

show = [("offered", "Offered volume (log10 dev.)"), ("n_bids", "Bids (log10 dev.)"),
        ("awarded", "Awarded volume (log10 dev.)"), ("n_rej", "Rejected bids (dev.)"),
        ("gap_above", "Gap to next rejected bid (log10)"), ("y", "Marginal price (log10 dev.)")]
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, (c, title) in zip(axes.flat, show):
    for d in ["downwards", "upwards"]:
        e = ev[(ev["direction"] == d) & (ev["stat"] == c)]
        ax.plot(e["k"], e["spike"], color=DIR_COLOR[d], marker="o", markersize=4, label=f"{d}: spike")
        ax.plot(e["k"], e["normal"], color=DIR_COLOR[d], linestyle=":", linewidth=1.5, label=f"{d}: normal")
    ax.axvline(0, color=GREY, linewidth=1); ax.axvspan(-4.3, -0.5, color="#f1f0ec", zorder=0)
    ax.set_title(title, loc="left", fontsize=10, color=INK)
for ax in axes[1]:
    ax.set_xlabel("weeks relative to the first week of the rise")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Weekly mFRR: bid book around price spikes (median deviation from recent level; shaded = known before gate closure)",
             x=0.01, ha="left", fontsize=11)
fig.tight_layout(); save(fig, "m3_prespike_event_study")


Is the pre-spike signal real? Mann-Whitney test and AUC of each statistic in the week **before the rise starts** (known before gate closure), both directions pooled, spikes vs normal weeks.

In [ ]:

def t1_deviation(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    for c in LOG:
        g[c] = np.log10(g[c])
    D = pd.DataFrame({c: g[c].shift(1) - g[c].shift(3).rolling(8, min_periods=5).median() for c in STATS})
    D["spike"] = g["rise_start"]
    keep = g["rise_start"] | (~g["in_episode"] & g["dy"].notna())
    return D[keep].dropna()

D = pd.concat([t1_deviation(g) for g in panels.values()])
rows = []
for c in STATS:
    s, n = D.loc[D["spike"], c], D.loc[~D["spike"], c]
    auc = roc_auc_score(D["spike"], D[c])
    rows.append({"stat (week before the rise)": c, "median spike": s.median(), "median normal": n.median(),
                 "p (Mann-Whitney)": mannwhitneyu(s, n).pvalue,
                 "AUC": max(auc, 1 - auc), "direction": "higher → spike" if auc > 0.5 else "lower → spike"})
print(f"{int(D['spike'].sum())} spikes / {int((~D['spike']).sum())} normal weeks")
pd.DataFrame(rows).set_index("stat (week before the rise)").sort_values("p (Mann-Whitney)").round(4)


## 4. Accepted vs rejected bids

`rel = log10(bid price / marginal price)` of the same auction: 0 = at the marginal price.

In [ ]:
B = offers[offers["mp"].notna()]
pd.concat({f: B[B["auction_frequency"] == f].groupby("accepted")["rel"].quantile([.1, .25, .5, .75, .9]).unstack()
           for f in ["weekly", "daily"]}).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=False)
bins = np.linspace(-1.5, 2.5, 81)
for ax, f in zip(axes, ["weekly", "daily"]):
    x = B[B["auction_frequency"] == f]
    ax.hist(x.loc[x["accepted"], "rel"].clip(-1.5, 2.5), bins=bins, color=C1, alpha=0.85, label="accepted")
    ax.hist(x.loc[~x["accepted"], "rel"].clip(-1.5, 2.5), bins=bins, color=C2, alpha=0.6, label="rejected")
    ax.axvline(0, color=INK, linewidth=1)
    ax.set_title(f"{f}: bid price relative to marginal, log10", loc="left", fontsize=10, color=INK)
    ax.set_xlabel("log10(bid / marginal)"); ax.set_ylabel("bids")
axes[0].legend()
fig.tight_layout(); save(fig, "m3_accepted_vs_rejected_price")

In [ ]:
pd.concat({f: B[B["auction_frequency"] == f].groupby("accepted")["offered_volume"].quantile([.1, .5, .9]).unstack()
           for f in ["weekly", "daily"]})

**Out-of-merit rejections.** The brief says a bid below the marginal price is *almost always* accepted. Here, in ~46 % of
auctions at least one cheaper bid is rejected. Is it because the bid was larger than the volume still needed (indivisible bid)?

In [ ]:
X = B.sort_values(["auction_id", "power_price"]).copy()
X["awarded_cheaper"] = X.groupby("auction_id")["awarded_volume"].cumsum() - X["awarded_volume"]
X["remaining_need"] = X.groupby("auction_id")["awarded_volume"].transform("sum") - X["awarded_cheaper"]
oom = X[~X["accepted"] & (X["power_price"] < X["mp"])]
both = B.groupby("auction_id")["accepted"].agg(["any", "all"])
n_two_sided = int((both["any"] & ~both["all"]).sum())
pd.Series({
    "auctions with accepted and rejected bids": n_two_sided,
    "auctions with a rejected bid below marginal": oom["auction_id"].nunique(),
    "share of those auctions": round(oom["auction_id"].nunique() / n_two_sided, 3),
    "rejected bids below marginal": len(oom),
    "share larger than remaining need (indivisible?)": round((oom["offered_volume"] > oom["remaining_need"]).mean(), 3),
    "partially accepted bids": int(((B["awarded_volume"] > 0) & (B["awarded_volume"] < B["offered_volume"])).sum()),
})

## 5. Can we follow a bidder? (comparable groups for Week 2)

No bidder IDs. Idea: use the bid **volume** as a fingerprint between consecutive weekly auctions of the same direction.

In [ ]:
print("most common bid sizes (MW):", offers["offered_volume"].value_counts().head(10).to_dict())

tracked, total = 0, 0
for d, g in offers[(offers["auction_frequency"] == "weekly") & ~offers["auction_id"].str.contains("KORR")].groupby("direction"):
    vols = g.groupby("auction_id")["offered_volume"].apply(lambda v: v.value_counts())
    ids = g.groupby("auction_id")["date"].first().sort_values().index
    for prev, cur in zip(ids[:-1], ids[1:]):
        vp, vc = vols.loc[prev], vols.loc[cur]
        tracked += len(set(vp[vp == 1].index) & set(vc[vc == 1].index))
        total += int(vc.sum())
print(f"bids with a volume unique in both consecutive weekly auctions: {tracked} / {total} = {tracked / total:.1%}")

→ Volume alone is a weak fingerprint (sizes are standardised: 5, 10, 20, 100 MW). Week 2 options:
match on (volume, nearest price) between consecutive auctions; or work at the level of the **supply curve**
(price quantiles of the order book) instead of individual bidders.

## 6. No-buy auctions (daily only — every weekly auction buys something)

In [ ]:
Dd = A[A["freq"] == "daily"].copy()
Dd["no_buy"] = Dd["y"].isna()
Dd.groupby([Dd["date"].dt.year, "direction"])["no_buy"].mean().unstack().round(3)

In [ ]:
m = Dd.groupby([pd.Grouper(key="date", freq="MS"), "direction"])["no_buy"].mean().unstack()
fig, ax = plt.subplots(figsize=(13, 3.2))
for d in m.columns:
    ax.plot(m.index, m[d], color=DIR_COLOR[d], label=d)
ax.set_title("Daily mFRR: share of auctions with nothing bought, per month", loc="left", fontsize=10, color=INK)
ax.legend(); fig.tight_layout(); save(fig, "m3_nobuy_share")

No-buy periods come in **whole delivery weeks** (runs of 42 blocks = 7 days × 6 blocks). Check the week-level structure and the link to the weekly auction of the same week (which closes before the daily ones):

In [ ]:
Dd["week"] = Dd["date"] - pd.to_timedelta(Dd["date"].dt.day_of_week, unit="D")
wk = Dd.groupby(["week", "direction"])["no_buy"].mean().rename("nb_share").reset_index()
print("week × direction cells that are all-or-nothing:", round(((wk["nb_share"] == 0) | (wk["nb_share"] == 1)).mean(), 3))

wa = W.groupby(["date", "direction"])["awarded"].sum().rename("weekly_awarded").reset_index().rename(columns={"date": "week"})
wk = wk.merge(wa, on=["week", "direction"])
wk = wk[wk["week"].dt.year != 2024]          # 2024 = near-total suspension, see below
rows = []
for d, g in wk.sort_values("week").groupby("direction"):
    g = g.copy()
    g["weekly_awarded_rel"] = np.log10(g["weekly_awarded"] / g["weekly_awarded"].shift(1).rolling(8, min_periods=4).median())
    g = g.dropna(subset=["weekly_awarded_rel"]); full = g["nb_share"] == 1
    rows.append({"direction": d, "weeks": len(g), "full no-buy weeks": int(full.sum()),
                 "weekly awarded vs usual (no-buy weeks)": g.loc[full, "weekly_awarded_rel"].median(),
                 "weekly awarded vs usual (other weeks)": g.loc[~full, "weekly_awarded_rel"].median(),
                 "AUC": roc_auc_score(full, g["weekly_awarded_rel"])})
pd.DataFrame(rows).set_index("direction").round(3)

In [ ]:
runs = []
for d, g in Dd.sort_values(["date", "block"]).groupby("direction"):
    r = (g["no_buy"] != g["no_buy"].shift()).cumsum()
    t = g.groupby(r).agg(no_buy=("no_buy", "first"), start=("date", "min"), end=("date", "max"), blocks=("no_buy", "size"))
    t["direction"] = d
    runs.append(t[t["no_buy"] & (t["blocks"] >= 200)])
pd.concat(runs)[["direction", "start", "end", "blocks"]]

## 7. Summary (Sprint 1, M3)

**Weekly spikes (Jean's question).**
- Spike = fast rise (≥ ×1.58 within 4 weeks) to a peak, then a decay (≥ 30 % of the rise within 8 weeks): **28 downwards + 19 upwards**. Median rise ×2.7 (down) / ×2.4 (up), reached in ~3 weeks; half of it is given back after a median of ~2 (down) / ~4 (up) weeks.
- Between spikes the price drifts slowly down (median weekly change < 0): a sawtooth.
- **Yes, the bid book changes before a spike.** In the week before the rise starts (known at gate closure), **offered volume (p = 1e-4, AUC 0.68) and the number of bids (p = 3e-4, AUC 0.66) are lower than usual**; fewer bids are rejected (p ≈ 0.01–0.02) and the cover ratio is lower (p = 0.04).
- **Awarded volume (Swissgrid's demand) does not move before spikes** (p = 0.27) → spikes look **supply-driven** (thinner book). For upward spikes, awarded volume rises slightly (~+5 %) *during* the rise, not before.
- Each signal alone is moderate (AUC ≤ 0.68): useful as features, not as a spike alarm.
- Long-run: the cover ratio fell from ~3–4 (2019) to ~1.7 (2021–24) while prices rose.

**Accepted vs rejected.**
- Weekly accepted bids sit tightly below the marginal (median −13 %); rejected bids median ~1.5× above. Daily books are much wider.
- ~46 % of two-sided auctions reject at least one bid *cheaper* than the marginal; only ~25 % of those are explained by "bid larger than remaining need" → clearing is not a pure merit order: indivisible step offers and joint TRL + aFRR cost minimisation (see `reports/m3_market_notes.md`).
- ~30 k bids are partially accepted.

**Following bidders.** Bid sizes are standardised; only ~8 % of weekly bids can be tracked by a unique volume → use (volume, price) matching or supply-curve-level analysis in Week 2.

**No-buy auctions.**
- Daily only. They come in whole delivery weeks (97 % of week × direction cells are all-or-nothing) and coincide with **higher-than-usual weekly awarded volume** of the same week (AUC 0.73 down / 0.84 up) → predictable from the weekly auction, which closes first.
- **2024: daily procurement almost suspended** (down: 2024-01-01 → 2024-12-01; up: 2024-02-05 → 2024-12-01). Resumes in December 2024; no no-buy auction in 2025.

**Limitations.** No bidder IDs; the spike definition is our choice (to confirm with Jean); 47 spikes is a small sample; 31 missing daily dates in 2024.